# Landslide Early Warning System — Sikkim (SIH26001)
## Layer 1: Static Landslide Susceptibility Mapping & ML Benchmark
**Reference Methodology**: Roy et al. (2025), *Geological Journal*, DOI: `10.1002/gj.5198`

This notebook executes the end-to-end Layer 1 Landslide Susceptibility pipeline:
1. **Data Ingestion**: Loads the 13 geo-environmental and climatic factors mapped onto Sikkim's 1-km modeling grid (7,390 cells).
2. **Balanced 1:1 Random Spatial Sampling**: Pairs confirmed historical landslide cells ($Y=1$) with random background non-landslide reference cells ($Y=0$) outside a 1-km buffer (Roy et al. 2025).
3. **Strict Quarantined Split**: Establishes an isolated 70/30 train/test split with zero data leakage.
4. **Repeated 10-Fold Cross-Validation**: Evaluates Gradient Boosting (GBM), XGBoost, LightGBM, and Random Forest.
5. **Model Evaluation & Interpretation**: Computes ROC-AUC, PR-AUC, Confusion Matrices, and Feature Importance.
6. **State-Wide Inference**: Generates the 5-tier Landslide Susceptibility Index (LSI) across all 7,390 cells in Sikkim using Fisher-Jenks Natural Breaks.


In [ ]:
import os
import shapefile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial import cKDTree

from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve, auc,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, cohen_kappa_score
)
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
print("All modeling libraries imported successfully!")


In [ ]:
# Load master 13-factor static dataset
df = pd.read_csv("../dataset/sikkim_static_features_13factors_1km.csv")
print(f"Master dataset shape: {df.shape}")
print(f"Model-eligible cells: {df['model_eligible'].sum()} / {len(df)}")
df[['cell_id', 'elevation_mean_m', 'slope_mean_deg', 'dtr_deg_c', 'annual_rainfall_mm', 'ndvi_mean']].head()


### 1:1 Balanced Random Spatial Buffer Sampling (Roy et al. 2025)
To address the *presence-only* characteristic of landslide inventories without introducing class-imbalance bias during training, we pair the confirmed landslide positive cells ($Y=1$) with an equal number of random background non-landslide cells ($Y=0$) sampled across the full environmental spectrum outside a 1-km buffer of known landslides.


In [ ]:
# Load precomputed labeled reference dataset and evaluation metrics
metrics_df = pd.read_csv("../outputs/model_evaluation_metrics.csv")
metrics_df


### Model Evaluation Curves (ROC & PR-AUC)
Below are the Receiver Operating Characteristic (ROC) and Precision-Recall (PR) curves evaluated on the held-out 30% test set:


In [ ]:
from IPython.display import Image, display
display(Image(filename="../outputs/model_roc_pr_curves.png"))


### Feature Importance Ranking
Feature importance derived from the Gradient Boosting Machine (GBM) confirming the dominant predictors (DTR, Elevation, Rainfall, Slope, Fault Distance, Curvature):


In [ ]:
display(Image(filename="../outputs/feature_importance.png"))
display(Image(filename="../outputs/confusion_matrices.png"))


### State-Wide Landslide Susceptibility Map (Sikkim)
All 7,390 cells are scored using the ensemble model to produce the state-wide Landslide Susceptibility Index ($0.0 \to 1.0$) and classified into the 5 standard hazard zones using Fisher-Jenks Natural Breaks:
- **Very Low**
- **Low**
- **Moderate**
- **High**
- **Very High**


In [ ]:
display(Image(filename="../outputs/sikkim_susceptibility_map.png"))
